# 02. AWQ: Activation-Aware Weight Quantization

- **Category:** 14_quantisation / 03_llm_ptq_algorithms
- **Lineage & Citations:** Lin et al. (2023, *AWQ: Activation-aware Weight Quantization for On-Device LLM Compression and Acceleration*).
- **Purpose:** Protect salient weight channels (0.1% to 1%) without mixed-precision memory overhead by mathematically scaling activations and weights via an equivalent channel transformation $\mathbf{Y} = (\mathbf{X}\mathbf{S}^{-1})(\mathbf{S}\mathbf{W})$.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [01. RTN vs GPTQ Hessian Error Compensation](01_rtn_vs_gptq_hessian_error_compensation.ipynb) $\to$ `🔗 Downstream Syntheses:` [03. SmoothQuant and Outlier Migration W8A8](03_smoothquant_and_outlier_migration_w8a8.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT keep salient channels in FP16 (which requires irregular hardware gather logic and slows down inference); do NOT set scale exponent $\alpha=1$ without weight clipping (unconstrained activation scaling blows up the weight dynamic range).

## Mathematical Foundations & Equivalent Transformation

1. **Equivalent Linear Transformation**:
   $$\mathbf{Y} = \mathbf{X}\mathbf{W} = (\mathbf{X}\mathbf{S}^{-1})(\mathbf{S}\mathbf{W}) = \widehat{\mathbf{X}}\widehat{\mathbf{W}}, \quad \mathbf{S} = \text{diag}(\mathbf{s})$$
2. **Channel Sensitivity Scaling**:
   $$\mathbf{s} = \frac{\mathbf{s}_{\text{act}}^\alpha}{\mathbf{s}_{\mathbf{w}}^{1 - \alpha}}, \quad \mathbf{s}_{\text{act}} = \frac{1}{N}\sum_{i=1}^N |\mathbf{X}_{i, :}|, \quad \mathbf{s}_{\mathbf{w}} = \max_j |W_{:, j}|$$
3. **Alpha Grid Search**:
   $$\alpha^* = \arg\min_{\alpha \in [0, 1]} \|\mathbf{X}\mathbf{W} - \mathbf{X}\mathbf{W}_{\text{eff}}(\alpha)\|_F^2$$
4. **Zero-Overhead Folding**:
   $$\gamma_{\text{new}} = \gamma \odot \mathbf{s}^{-1} \implies \text{absorbed into previous LayerNorm/RMSNorm}$$

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

torch.manual_seed(42)
r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
WEIGHTS_PATH = r / "weights/14_quantisation/03_llm_ptq_algorithms/02_awq_activation_aware_weight_quantization.safetensors"
weights = load_file(str(WEIGHTS_PATH))
print(f"Loaded {len(weights)} tensors from {WEIGHTS_PATH.name}")

In [ ]:
def rtn_quantize_per_column(weight: torch.Tensor, qmin: float = -8.0, qmax: float = 7.0) -> tuple[torch.Tensor, torch.Tensor]:
    scale = torch.max(torch.abs(weight), dim=0, keepdim=True).values / qmax
    scale = torch.where(scale == 0.0, torch.ones_like(scale), scale)
    q = torch.clamp(torch.round(weight / scale), qmin, qmax)
    return q * scale, scale

def awq_compute_channel_scales(s_act: torch.Tensor, s_w: torch.Tensor, alpha: float) -> torch.Tensor:
    eps = 1e-6
    s_act_safe = torch.clamp(s_act, min=eps)
    s_w_safe = torch.clamp(s_w, min=eps)
    s = (s_act_safe ** alpha) / (s_w_safe ** (1.0 - alpha))
    norm = torch.sqrt(torch.max(s) * torch.min(s) + eps)
    return s / norm

def awq_search_optimal_scale(x_calib: torch.Tensor, weight: torch.Tensor, alpha_grid: list[float] | None = None) -> tuple[torch.Tensor, float, torch.Tensor]:
    if alpha_grid is None: alpha_grid = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
    s_act = torch.mean(torch.abs(x_calib), dim=0)
    s_w = torch.max(torch.abs(weight), dim=1).values
    y_ref = x_calib @ weight
    best_loss = float("inf")
    best_alpha, best_scale, best_w_eff = 0.5, s_act, weight
    for alpha in alpha_grid:
        s = awq_compute_channel_scales(s_act, s_w, alpha)
        w_scaled = weight * s.unsqueeze(1)
        w_scaled_q, _ = rtn_quantize_per_column(w_scaled)
        w_eff = w_scaled_q / s.unsqueeze(1)
        loss = float(torch.mean((y_ref - x_calib @ w_eff) ** 2))
        if loss < best_loss:
            best_loss, best_alpha, best_scale, best_w_eff = loss, alpha, s, w_eff
    return best_scale, best_alpha, best_w_eff

In [ ]:
# 1. Linear anchor parity
y = weights["test_input"] @ weights["weight"] + weights["bias"]
assert float(torch.max(torch.abs(y - weights["test_output"]))) < 1e-5

# 2. RTN baseline parity
w_rtn, col_scales = rtn_quantize_per_column(weights["w_orig"])
assert float(torch.max(torch.abs(w_rtn - weights["w_rtn"]))) < 1e-5

# 3. Channel sensitivity metrics parity
s_act = torch.mean(torch.abs(weights["x_calib"]), dim=0)
s_w = torch.max(torch.abs(weights["w_orig"]), dim=1).values
assert float(torch.max(torch.abs(s_act - weights["s_act"]))) < 1e-5
assert float(torch.max(torch.abs(s_w - weights["s_w"]))) < 1e-5

# 4. AWQ search parity
alpha_grid = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
best_scale, best_alpha, w_awq = awq_search_optimal_scale(weights["x_calib"], weights["w_orig"], alpha_grid)
assert abs(best_alpha - float(weights["best_alpha"][0])) < 1e-5
assert float(torch.max(torch.abs(best_scale - weights["best_scale"]))) < 1e-5
assert float(torch.max(torch.abs(w_awq - weights["w_awq"]))) < 1e-5
print("All AWQ parity assertions verified against golden checkpoint!")

In [ ]:
# Comparative SQNR Benchmark: RTN vs AWQ under Outlier Channels
x_eval = weights["x_calib"]
y_true = x_eval @ weights["w_orig"]
sig_pwr = float(torch.mean(y_true ** 2))

def eval_sqnr(w_q: torch.Tensor) -> float:
    y_pred = x_eval @ w_q
    mse = float(torch.mean((y_true - y_pred) ** 2))
    return 10.0 * torch.log10(torch.tensor(sig_pwr / (mse + 1e-12))).item()

sqnr_rtn = eval_sqnr(weights["w_rtn"])
sqnr_awq = eval_sqnr(weights["w_awq"])

print(f"RTN SQNR: {sqnr_rtn:.2f} dB")
print(f"AWQ SQNR: {sqnr_awq:.2f} dB (gain: +{sqnr_awq - sqnr_rtn:.2f} dB)")
assert sqnr_awq > sqnr_rtn + 5.0, "AWQ must protect outlier channels, gaining >5 dB SQNR over RTN!"

In [ ]:
# Latency Microbenchmark: Fused Dequantization and Channel Rescaling
d_in, d_out = 1024, 1024
w_q = torch.randint(-8, 7, (d_in, d_out)).float()
c_scale = torch.randn(1, d_out).abs() * 0.1
s_channel = torch.randn(d_in, 1).abs() + 0.5

t0 = time.perf_counter()
for _ in range(50):
    w_eff = (w_q * c_scale) / s_channel
t_awq = (time.perf_counter() - t0) / 50 * 1e3

print(f"AWQ Dequant & Channel Rescale latency: {t_awq:.3f} ms")
assert t_awq < 10.0, "Fused channel dequantization must execute in <10 ms on CPU"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Protection Without Mixed Precision**: AWQ scales salient channels to shield them from quantization noise while preserving a uniform 4-bit representation.
2. **Grid Search Optimization**: Searching $\alpha \in [0, 1]$ balances activation sensitivity against weight dynamic range expansion.
3. **Zero Inference Overhead**: Channel scales are folded into preceding LayerNorm/RMSNorm weights, incurring zero latency.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [02_awq_activation_aware_weight_quantization.md](../../../markdown/14_quantisation/03_llm_ptq_algorithms/02_awq_activation_aware_weight_quantization.md)
- **PyTorch 2.x Module:** [02_awq_activation_aware_weight_quantization.py](../../../pytorch2/14_quantisation/03_llm_ptq_algorithms/02_awq_activation_aware_weight_quantization.py)
- **Functional JAX Module:** [02_awq_activation_aware_weight_quantization.py](../../../jax/14_quantisation/03_llm_ptq_algorithms/02_awq_activation_aware_weight_quantization.py)
- **Native Rust Candle Track:** [02_awq_activation_aware_weight_quantization.rs](../../../candle/14_quantisation/03_llm_ptq_algorithms/02_awq_activation_aware_weight_quantization.rs)
- **Hardware Kernel (CPU / WGSL):** [02_awq_activation_aware_weight_quantization.rs](../../../kernels/14_quantisation/03_llm_ptq_algorithms/02_awq_activation_aware_weight_quantization.rs)
- **Golden SafeTensors:** [02_awq_activation_aware_weight_quantization.safetensors](../../../weights/14_quantisation/03_llm_ptq_algorithms/02_awq_activation_aware_weight_quantization.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 01. RTN vs GPTQ Hessian Error Compensation](01_rtn_vs_gptq_hessian_error_compensation.ipynb) | [Master Index](../../../README.md#14_quantisation) | [Next: 03. SmoothQuant and Outlier Migration W8A8](03_smoothquant_and_outlier_migration_w8a8.ipynb) &rarr;